In [22]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier

import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_20_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding[nuc] for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))
vectorizer = CountVectorizer()
X_kmers_sparse = vectorizer.fit_transform(base["kmers"])
X_kmers = X_kmers_sparse.toarray()


gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)

train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

X_train_kmers, X_test_kmers = X_kmers[train_id], X_kmers[test_id]

y_train, y_test = y[train_id], y[test_id]

model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 300, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 300, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_score_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_score_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

print(f'AUC-ROC One-Hot: {auc_score_oh:.4f}')
print(f'AUC-ROC k-mers: {auc_score_kmers:.4f}')

/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


AUC-ROC One-Hot: 0.5352
AUC-ROC k-mers: 0.5951


/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
